In [ ]:
import json

import torch 
from selfcheckgpt.modeling_selfcheck import SelfCheckNLI
import spacy
import time

nlp = spacy.load("autodl-tmp/en_core_web_sm")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
selfcheck_nli = SelfCheckNLI(device=device, nli_model='autodl-tmp/deberta-v3-large-mnli') # set device to 'cuda' if GPU is available

with open('GLOFer_dl19/hypothesis_documents_dl19_5', 'r') as file:
    hypothesis_documents_dl19 = json.load(file)

for p in range(len(hypothesis_documents_dl19)):
    hypothesis_documents=[row[2:] for row in hypothesis_documents_dl19][p] 
    documents_nli=[]
    for l in range(5):
        document_nli=[]
        passage=hypothesis_documents[l]
        passage =  '"' * 3 + passage + '"' * 3 
        passage=passage.replace("\n", " ").replace("\t", " ").strip()
        sentences = [sent for sent in nlp(passage).sents] # List[spacy.tokens.span.Span]
        sentences = [sent.text.strip() for sent in sentences if len(sent) > 3]
        hypothesis_documents_left=[]
        for q in range(5):
            if l!=q:
               c=hypothesis_documents[q]
               c =  '"' * 3 + c + '"' * 3
               c=c.replace("\n", " ").replace("\t", " ").strip()
               hypothesis_documents_left.append(c) 
        sample1=hypothesis_documents_left[0]
        sample2=hypothesis_documents_left[1]
        sample3=hypothesis_documents_left[2]
        sample4=hypothesis_documents_left[3]
        sent_scores_nli = selfcheck_nli.predict(
            sentences = sentences,                          # list of sentences
            sampled_passages = [sample1, sample2, sample3, sample4], # list of sampled passages
        )
        for k in range(len(sentences)):
            sent_nli=[sentences[k],sent_scores_nli[k]]
            document_nli.append(sent_nli)
        documents_nli.append(document_nli)
    hypothesis_documents_dl19[p][2:]=documents_nli 
    
    with open('GLOFer_dl19/hypothesis_documents_dl19_5_NLI', 'w') as file:
         json.dump(hypothesis_documents_dl19, file)

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from numpy import *
from scipy.stats import entropy
import spacy

llm_model="autodl-tmp/LLM-Research/Meta-Llama-3-8B-Instruct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
nlp = spacy.load("autodl-tmp/en_core_web_sm")

tokenizer= AutoTokenizer.from_pretrained(llm_model)
model = AutoModelForCausalLM.from_pretrained(llm_model)
model = model.eval()
model = model.to(device)


for p in range(len(hypothesis_documents_dl19)):
    hypothesis_documents=[row[2:] for row in hypothesis_documents_dl19][p]
    documents_entAtt_probs=[]
    for l in range(5):
        document_entAtt_probs=[]
        passage=hypothesis_documents[l]
        passage =  "\"" * 3 + passage + "\"" * 3
        passage=passage.replace("\n", " ").strip()
        sentences = [sent for sent in nlp(passage).sents] # List[spacy.tokens.span.Span]
        sentences = [sent.text.strip() for sent in sentences if len(sent) > 3]
        for i in range(len(sentences)):
            input = tokenizer(sentences[i], return_tensors="pt").to(device)
            output1=model(input.input_ids,output_attentions=True)
            logits = output1.logits
            prob = torch.softmax(logits, dim=-1)[0]
            probcpu=prob.cpu().detach().numpy()
            entropies=entropy(prob.cpu().detach().numpy(), base=2,axis=-1)
            attentions=output1.attentions
            attentions = attentions[-1][0]
            mean_atten = torch.sum(attentions, dim=1)
            mean_atten = torch.mean(mean_atten, dim=0)
            for k in range(mean_atten.shape[0]):
                mean_atten[k] /= (mean_atten.shape[0] - k)
            mean_atten=mean_atten.cpu().detach().numpy()
            sen_entropyAtten=entropies[1:]@mean_atten[1:]/len(mean_atten[1:])
            sen_probs=[]
            sen_tokens_prob=[]
            for k in range(input.input_ids.size()[1]-1):
                sen_probs.append(probcpu[k+1,input.input_ids[0][k+1]].astype(float))
            sent_entAtt_probs=[sentences[i],sen_entropyAtten,sen_probs]#,sen_tokens_prob]
            document_entAtt_probs.append(sent_entAtt_probs)
        documents_entAtt_probs.append(document_entAtt_probs)
    hypothesis_documents_dl19[p][2:]=documents_entAtt_probs


with open('GOLFer_dl19hypothesis_documents_dl19_5_entAtt_probs', 'w') as file:
    json.dump(hypothesis_documents_dl19, file)    


In [ ]:
import json
 
with open('GOLFer_dl19hypothesis_documents_dl19_5_NLI', 'r') as file:
    documents_nli_dl19 = json.load(file)

with open('GOLFer_dl19hypothesis_documents_dl19_5_entAtt_probs', 'r') as file:
    documents_entAtt_probs_dl19 = json.load(file)
    
#根据乘积分数
for p in range(len(documents_nli_dl19)):
    print(p)
    documents_nli=[row[2:] for row in documents_nli_dl19][p] 
    documents_entAtt_probs=[row[2:] for row in documents_entAtt_probs_dl19][p] 
    qualified_documents=[]
    for l in range(5):
        document_nli=documents_nli[l]
        document_entAtt_probs=documents_entAtt_probs[l]
        qualified_document_withscore=[[b[0], b[2], a[1]*b[1]] for a, b in zip(document_nli, document_entAtt_probs) if a[1]*b[1]<0.8]
        num_del_sents=len(document_nli)-len(qualified_document_withscore)
        print(num_del_sents)
        qualified_document=''
        qualified_document_prob = 0
        num_tokens=0
        for j in range(len(qualified_document_withscore)):
            qualified_document+=qualified_document_withscore[j][0]
            for k in range(len(qualified_document_withscore[j][1])):
                num_tokens+=1
                qualified_document_prob+=qualified_document_withscore[j][1][k]
        if num_tokens!=0:
            qualified_document_prob=qualified_document_prob/num_tokens
        else:
            qualified_document_prob=0
        qualified_documents.append([qualified_document,qualified_document_prob])
    documents_nli_dl19[p][2:]=qualified_documents[0:]

with open('GOLFer_dl19/hypothesis_documents_dl19_5_qualified', 'w') as file:
     json.dump(documents_nli_dl19, file)
    